# Integrate Google Agentspace with Gen AI Toolbox

This notebook provides a working example of integrating [Google Agentspace](https://cloud.google.com/products/agentspace?e=48754805&hl=en) with Google Cloud databases via [Gen AI Toolbox](https://github.com/googleapis/genai-toolbox). While this notebook uses Spanner as the source database, you can leverage the same pattern to integrate with [any database](https://googleapis.github.io/genai-toolbox/resources/sources/#available-sources) that Gen AI Toolbox supports.

## Basic Setup

## Pre-requisites

You will need to be allow-listed for Agentspace access before you can run the final steps in this notebook, so please work with your account team to gain access to the preview feature. However, you can run all of the steps up to and including the creation of an Agentspace Conversational App that integrates with Spanner via Toolbox without being allow-listed.

### Install dependencies

In [ ]:
%pip install \
    langgraph==0.3.21 \
    langchain-google-vertexai==2.0.18 \
    toolbox-langchain==0.1.0 \
    google.cloud==0.34.0 \
    google-cloud-discoveryengine==0.13.8 \
    --quiet

### Authenticate to Google Cloud within Colab
If you're running this on google colab notebook, you will need to Authenticate as an IAM user.

In [ ]:
from google.colab import auth

auth.authenticate_user()

### Define Notebook Parameters

In [ ]:
# @markdown Update the parameters below to match your environment.
# @markdown > Note: You can leave bucket names empty to create new GCS buckets.

# Please fill in these values.
project_id = "your-project"  # @param {type:"string"}
region = "your-region"  # @param {type:"string"}
vpc = "your-vpc"  # @param {type:"string"}

# Optionally change these values
spanner_instance_id = "spanner-instance"  # @param {type:"string"}
spanner_database_id = "ecom-database"  # @param {type:"string"}
spanner_table_id = "products"  # @param {type:"string"}
spanner_avro_export_location = "gs://pr-public-demo-data/spanner-retail-demo/products-instance-orders-database/"  # @param {type:"string"}

# Do not change these values
spanner_session = None  # Global Spanner session variable
project_number = ! gcloud projects describe {project_id} --format='value(projectNumber)'
project_number = project_number[0]


### Connect Your Google Cloud Project

In [ ]:
# Configure gcloud.
!gcloud config set project {project_id}

### Enable APIs

In [ ]:
!gcloud services enable spanner.googleapis.com \
                        run.googleapis.com \
                        cloudbuild.googleapis.com \
                        artifactregistry.googleapis.com \
                        iam.googleapis.com \
                        secretmanager.googleapis.com \
                        discoveryengine.googleapis.com

### Configure Logging

In [ ]:
import logging
import sys

# Configure the root logger to output messages with INFO level or above
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)s[%(levelname)5s][%(name)14s] - %(message)s',  datefmt='%H:%M:%S', force=True)

### Helper Functions

#### rest_api_helper()

In [ ]:
import requests
import google.auth
import json

# Get an access token based upon the current user
creds, _ = google.auth.default()
authed_session = google.auth.transport.requests.AuthorizedSession(creds)
access_token=creds.token

if project_id:
  authed_session.headers.update({"x-goog-user-project": project_id}) # Required to workaround a project quota bug

def rest_api_helper(
    session: requests.Session,
    url: str,
    http_verb: str,
    request_body: dict = None,
    params: dict = None
  ) -> dict:
  """Calls a REST API using a pre-authenticated requests Session."""

  headers = {"Content-Type": "application/json"}

  try:

    if http_verb == "GET":
      response = session.get(url, headers=headers, params=params)
    elif http_verb == "POST":
      response = session.post(url, json=request_body, headers=headers, params=params)
    elif http_verb == "PUT":
      response = session.put(url, json=request_body, headers=headers, params=params)
    elif http_verb == "PATCH":
      response = session.patch(url, json=request_body, headers=headers, params=params)
    elif http_verb == "DELETE":
      response = session.delete(url, headers=headers, params=params)
    else:
      raise ValueError(f"Unknown HTTP verb: {http_verb}")

    # Raise an exception for bad status codes (4xx or 5xx)
    response.raise_for_status()

    # Check if response has content before trying to parse JSON
    if response.content:
        return response.json()
    else:
        return {} # Return empty dict for empty responses (like 204 No Content)

  except requests.exceptions.RequestException as e:
      # Catch potential requests library errors (network, timeout, etc.)
      # Log detailed error information
      print(f"Request failed: {e}")
      if e.response is not None:
          print(f"Request URL: {e.request.url}")
          print(f"Request Headers: {e.request.headers}")
          print(f"Request Body: {e.request.body}")
          print(f"Response Status: {e.response.status_code}")
          print(f"Response Text: {e.response.text}")
          # Re-raise a more specific error or a custom one
          raise RuntimeError(f"API call failed with status {e.response.status_code}: {e.response.text}") from e
      else:
          raise RuntimeError(f"API call failed: {e}") from e
  except json.JSONDecodeError as e:
      print(f"Failed to decode JSON response: {e}")
      print(f"Response Text: {response.text}")
      raise RuntimeError(f"Invalid JSON received from API: {response.text}") from e



## Setup Spanner

### Define Spanner Helper Functions

In [ ]:
def get_spanner_sessions(project_id = project_id, instance_id = spanner_instance_id, database_id = spanner_database_id):
  url = f"https://spanner.googleapis.com/v1/projects/{project_id}/instances/{instance_id}/databases/{database_id}/sessions"
  response = rest_api_helper(authed_session, url, "GET")
  return response

In [ ]:
# https://cloud.google.com/spanner/docs/reference/rest/v1/projects.instances.databases.sessions/create
def create_spanner_session(project_id = project_id, instance_id = spanner_instance_id, database_id = spanner_database_id):

  # Create a new session
  url = f"https://spanner.googleapis.com/v1/projects/{project_id}/instances/{instance_id}/databases/{database_id}/sessions"
  params = {
      "database": f"projects/{project_id}/instances/{instance_id}/databases/{database_id}"
  }
  response = rest_api_helper(authed_session, url, "POST", {}, params)
  return response['name']

In [ ]:
# https://cloud.google.com/spanner/docs/reference/rest/v1/projects.instances.databases.sessions/delete
def close_spanner_session(session, project_id = project_id, instance_id = spanner_instance_id, database_id = spanner_database_id):
  url = f"https://spanner.googleapis.com/v1/{session}"
  response = rest_api_helper(authed_session, url, "DELETE", {}, {"name": f"{session}"})
  return response

In [ ]:
import pandas as pd

def run_spanner_query(sql, database_id = spanner_database_id, query_options=None, create_new_session=False):
  """
  Runs a Spanner query and returns the result.

  Args:
      sql: The SQL query to execute.
      database_id: The database to query.
      query_options: (Optional) A dictionary of advanced query options.
                    See https://cloud.google.com/spanner/docs/reference/rest/v1/projects.instances.databases.sessions/executeSql#queryoptions
                    for available options.
      create_new_session: Defines whether to run the query in a new session.

  Returns:
      A dictionary containing the query results.

  Ref:
      https://cloud.google.com/spanner/docs/reference/rest/v1/projects.instances.databases.sessions/executeSql
  """
  # Ensure a spanner_session exists
  global spanner_session
  if not spanner_session or create_new_session == True:
    spanner_session = create_spanner_session()

  # Initialize response vars
  commit_response = ""
  response = ""

  # Construct the request URL
  uri = f"https://spanner.googleapis.com/v1/{spanner_session}:executeSql"

  # Set transaction type (readOnly/readWrite) and transaction object with commit type (begin/singleUse)
  transaction_type = "readWrite" if any(x in sql.lower() for x in ["insert", "update", "delete"]) else "readOnly"
  transaction = {"begin": {"readWrite": {}}} if transaction_type == "readWrite" else {"singleUse": {"readOnly": {}}}

  # Construct the request
  request_body = {
      "sql": sql,
      "transaction": transaction
  }
  params = {
      "session": spanner_session
  }

  if query_options:
      request_body["queryOptions"] = query_options

  try:
    # Make the request
    response = rest_api_helper(authed_session, uri, "POST", request_body=request_body, params = params)

  except RuntimeError as e:
    if "Session not found" in str(e):
      print(f"Session not found. Creating a new session and retrying the query...")
      return run_spanner_query(sql, database_id, query_options, create_new_session=True)  # Retry with a new session
    else:
      raise  # Re-raise the exception if it's not a "Session not found" error

  # Commit transaction if read/write
  if transaction_type == "readWrite":
      uri = f"https://spanner.googleapis.com/v1/{spanner_session}:commit"
      params = {
          "session": spanner_session
      }
      commit_response = rest_api_helper(authed_session, uri, "POST", {"transactionId": response['metadata']['transaction']['id']}, params)
      print(f"commit_response: {commit_response}")

  # Return a DataFrame if type is SELECT, WITH, or GRAPH
  if transaction_type == 'readOnly':
    columns = [field.get('name', 'unnamed_column') for field in response['metadata']['rowType']['fields']]

    # Create DataFrame from rows
    if 'rows' in response:
      df = pd.DataFrame(response['rows'], columns=columns)
      return df
    else:
      return response

  else:
    # Return the query results
    return response

In [ ]:
import time

def run_spanner_ddl(ddl_array, project_id = project_id, instance_id = spanner_instance_id, database_id = spanner_database_id):
  # https://cloud.google.com/spanner/docs/reference/rest/v1/projects.instances.databases.tables/create#try-it

  uri = f"https://spanner.googleapis.com/v1/projects/{project_id}/instances/{instance_id}/databases/{database_id}/ddl"
  http_verb = "PATCH"
  request_body = {
      "statements": ddl_array
  }

  response = rest_api_helper(authed_session, uri, http_verb, request_body)

  operation_name = response['name']
  uri = f"https://spanner.googleapis.com/v1/{operation_name}"

  while True:
    response = rest_api_helper(authed_session, uri, "GET", {})
    if response.get("done", False):
      if response.get("error"):
        print(response.get("error"))
      else:
        print("Operation completed successfully.")
      break
    else:
      print("Operation not completed yet.")
      time.sleep(2)


### Create Spanner Instance

In [ ]:
# Create the Spanner instance
# This notebook creates a paid instance. 90-day Free trial available once per project lifecycle
# https://cloud.google.com/spanner/docs/reference/rest/v1/projects.instances/create
url = f"https://spanner.googleapis.com/v1/projects/{project_id}/instances"
http_verb = "POST"
request_body = {
    "instance": {
        "config": f"projects/{project_id}/instanceConfigs/regional-us-central1",
        "displayName": f"{spanner_instance_id}",
        "edition": "ENTERPRISE",
        "processingUnits": 100,

        # OPTIONAL: Define nodeCount instead of processingUnits or autoscalingConfig.
        #"nodeCount": 1,

        # OPTIONAL: Define autoscalingConfig instead of nodeCount or processingUnits.
        #"autoscalingConfig": {
        #  "autoscalingLimits": {
        #    "minProcessingUnits": 1000,
        #    "maxProcessingUnits": 2000
        #  },
        #  "autoscalingTargets": {
        #    "highPriorityCpuUtilizationPercent": 80,
        #    "storageUtilizationPercent": 80
        #  }
        #}
    },
    "instanceId": f"{spanner_instance_id}"
}

response = rest_api_helper(authed_session, url, http_verb, request_body)
response

In [ ]:
# Create the Spanner database
# https://cloud.google.com/spanner/docs/reference/rest/v1/projects.instances.databases/create
url = f"https://spanner.googleapis.com/v1/projects/{project_id}/instances/{spanner_instance_id}/databases"
http_verb = "POST"
request_body = {
    "createStatement": f"CREATE DATABASE `{spanner_database_id}`",
    "databaseDialect": "GOOGLE_STANDARD_SQL"
}

response = rest_api_helper(authed_session, url, http_verb, request_body)
response

### Add Required Permissions for Database Import

In [ ]:
roles_array = [
    "roles/spanner.viewer",
    "roles/dataflow.worker",
    "roles/storage.admin",
    "roles/spanner.databaseReader",
    "roles/spanner.databaseAdmin",
]

for r in roles_array:
  ! gcloud projects add-iam-policy-binding {project_id} \
      --member="serviceAccount:{project_number}-compute@developer.gserviceaccount.com" \
      --role="{r}"


### Enable Vertex AI Integration

In [ ]:
# Create and Embeddings Model and LLM Model
# Ref: https://codelabs.developers.google.com/codelabs/spanner-getting-started-vector-search#3
#      https://cloud.google.com/spanner/docs/ml-tutorial-embeddings
ddl_array = []
ddl_array.append(f"""CREATE MODEL IF NOT EXISTS LLMModel INPUT(
prompt STRING(MAX),
) OUTPUT(
content STRING(MAX),
) REMOTE OPTIONS (
endpoint = '//aiplatform.googleapis.com/projects/{project_id}/locations/us-central1/publishers/google/models/gemini-2.0-flash-001',
default_batch_size = 1
)
""")

ddl_array.append(f"""CREATE MODEL IF NOT EXISTS EmbeddingsModel INPUT(
  content STRING(MAX),
  ) OUTPUT(
  embeddings STRUCT<statistics STRUCT<truncated BOOL, token_count FLOAT64>, values ARRAY<FLOAT64>>,
  ) REMOTE OPTIONS (
  endpoint = '//aiplatform.googleapis.com/projects/{project_id}/locations/us-central1/publishers/google/models/text-embedding-005'
  )
""")

result = run_spanner_ddl(ddl_array)
result

### Test the Models

> NOTE: It may take a minute or two for the integration to complete in the background before the tests below will work.

In [ ]:
sql = """
SELECT embeddings.values
  FROM ML.PREDICT(
    MODEL EmbeddingsModel,
    (SELECT 'This is a test string that will be converted to an embedding' as content))
"""

run_spanner_query(sql)

In [ ]:
sql = """SELECT *
FROM ML.PREDICT(
MODEL LLMModel,
(   SELECT
'What are Google Agentspace and Gen AI Toolbox?' AS prompt),
STRUCT(256 AS maxOutputTokens))"""

run_spanner_query(sql)

## Load Spanner Data

The sample dataset is based on [theLook eCommerce](https://console.cloud.google.com/bigquery/analytics-hub/discovery/projects/1057666841514/locations/us/dataExchanges/google_cloud_public_datasets_17e74966199/listings/thelook_ecommerce) synthetic eCommerce and Digital Marketing data.

### Kick Off Import Job

In [ ]:
# Kick off the Dataflow import job
result = ! gcloud dataflow jobs run import-spanner \
    --gcs-location='gs://dataflow-templates-{region}/latest/GCS_Avro_to_Cloud_Spanner' \
    --region={region} \
    --parameters='instanceId={spanner_instance_id},databaseId={spanner_database_id},inputDir={spanner_avro_export_location}' \
    --network={vpc}

# Get id of Dataflow job from result
job_id = ""
for item in result:
  if item.startswith('id'):
    job_id = item.split()[1]

# Show result
result

### Wait for Import Completion

Grab a cup of coffee or tea. This step will take about 15 minutes.

In [ ]:
import time

# Define Helper Function
def wait_for_dataflow_job(id: str):
  # Check status of Dataflow job
  job_state = ! gcloud dataflow jobs describe {job_id} --region={region} --format='value(currentState)'

  # Wait until Dataflow job is complete, checking status every 10 seconds
  while job_state[0] in ['JOB_STATE_RUNNING', 'JOB_STATE_PENDING']:
    print(f"Dataflow job {job_id} is in state: {job_state[0]}")
    time.sleep(10)
    job_state = ! gcloud dataflow jobs describe {job_id} --region={region} --format='value(currentState)'

  # Show final Dataflow job state
  print(f"Dataflow job {job_id} final state: {job_state[0]}")
  return job_state[0]

# Wait for job to complete
wait_for_dataflow_job(job_id)

## Generate Vector Embeddings

### Add Embedding Columns

This section will walk you through generating embeddings of the product data in theh Spanner database. This will allow us to run flexible natual language queries on the products data in our Agentspace and Gen AI Toolbox agents.

Reference: https://cloud.google.com/spanner/docs/backfill-embeddings

In [ ]:
# Add embedding column
ddl_array = []

ddl_array.append("ALTER TABLE products ADD COLUMN embedding ARRAY<FLOAT64>")
ddl_array.append("ALTER TABLE products ADD COLUMN embedding_model_version STRING(256)")

run_spanner_ddl(ddl_array)

### Backfill Vector Embeddings

This step will take around 10 minutes.

> NOTE: You may see errors in the log like, "Database schema probably changed during transaction, retry may succeed". This is an artifact of the recent schema changes and asynchronous foreign key operations. The tenacity library should handle retries for you automatically.

In [ ]:
from tenacity import retry, wait_exponential, stop_after_attempt, before_sleep_log, retry_if_exception

def retry_condition(error):
  error_string = str(error)
  print(error_string)

  retry_errors = [
      "Database schema has changed",
      "Transaction aborted. Database schema probably changed during transaction, retry may succeed.",
      "Transaction was aborted. It was wounded by a higher priority transaction",
      "Transaction was aborted."
      # Add more error messages here as needed
  ]

  for retry_error in retry_errors:
    if retry_error in error_string:
      print("Retrying...")
      return True

  return False

In [ ]:
@retry(wait=wait_exponential(multiplier=1, min=1, max=60), stop=stop_after_attempt(10), retry=retry_if_exception(retry_condition), before_sleep=before_sleep_log(logging.getLogger(), logging.INFO))
def backfill_embeddings():
  count_sql = "SELECT COUNT(*) AS to_embed FROM products WHERE products.embedding IS NULL"
  count_result = run_spanner_query(count_sql)
  rows_left = int(count_result.values[0][0])

  while rows_left > 0:
    sql = """UPDATE products
    SET
      products.embedding = (
        SELECT embeddings.values
        FROM SAFE.ML.PREDICT(
          MODEL EmbeddingsModel,
          (SELECT CONCAT('Name: ', name, ' \\nCategory: ', category, ' \\nBrand: ', brand, ' \\nDepartment: ', department) AS content)
        ) @{remote_udf_max_rows_per_rpc=200}
      ),
      products.embedding_model_version = 'text-embedding-005'
    WHERE products.id IN (SELECT p_sub.id
        FROM products AS p_sub
        WHERE p_sub.embedding IS NULL
        LIMIT 200)
    """

    run_spanner_query(sql)

    count_sql = "SELECT COUNT(*) AS to_embed FROM products WHERE products.embedding IS NULL"
    count_result = run_spanner_query(count_sql)
    rows_left = int(count_result.values[0][0])
    print(f"{rows_left} more rows left to embed...")

  print("Embedding process is complete")

backfill_embeddings()

### Test Vector Embedding Query

In [ ]:
search_phrase = "Luxury items for men"

sql = f"""WITH e AS (
    SELECT embeddings.values
    FROM ML.PREDICT(
    MODEL EmbeddingsModel, (
            SELECT '{search_phrase}' as content
        )
    )
)
SELECT COSINE_DISTANCE(
    products.embedding,
    e.values
  ) as dist,
  id,
  name,
  brand,
  department
FROM products, e
ORDER BY dist
LIMIT 5;
"""

run_spanner_query(sql)

## Setup Toolbox

### Deploy Gen AI Toolbox to Cloud Run

Reference: https://github.com/googleapis/genai-toolbox/blob/main/docs/en/how-to/deploy_toolbox.md

#### Add Required Permissions to Deploy Gen AI Toolbox

In [ ]:
roles_array = [
    "roles/iam.serviceAccountCreator",
    "roles/secretmanager.admin",
    "roles/run.developer",
    "roles/iam.serviceAccountUser",
]

for r in roles_array:
  ! gcloud projects add-iam-policy-binding {project_id} \
      --member="serviceAccount:{project_number}-compute@developer.gserviceaccount.com" \
      --role="{r}"


#### Create a Service Account and Add Permissions to Run the Gen AI Toolbox Service

In [ ]:
! gcloud iam service-accounts create toolbox-identity

roles_array = [
    "roles/secretmanager.secretAccessor",
    "roles/spanner.viewer",
    "roles/spanner.databaseReader",
    "roles/spanner.databaseAdmin",
]

for r in roles_array:
  ! gcloud projects add-iam-policy-binding {project_id} \
      --member serviceAccount:toolbox-identity@{project_id}.iam.gserviceaccount.com \
      --role="{r}"

#### Configure tools.yaml

In [ ]:
# Reference: https://googleapis.github.io/genai-toolbox/resources/sources/spanner/
#            https://googleapis.github.io/genai-toolbox/resources/tools/
#            https://googleapis.github.io/genai-toolbox/resources/tools/spanner-sql/

import os
import json

tools_config = {
  "sources": {
    "spanner-ecom-source": {
        "kind": "spanner",
        "project": f"{project_id}",
        "instance": f"{spanner_instance_id}",
        "database": f"{spanner_database_id}",
        "dialect": "googlesql"
      }
    },
  "tools": {
    "get_sales_data": {
      "kind": "spanner-sql",
      "source": "spanner-ecom-source",
      "description": "Use this tool to calculate sales for the last day, week, month, and year.",
      "statement": """-- Daily, weekly, monthly, yearly sales:
-- Define the relevant time boundaries based on the *fixed* date '2024-12-03'
WITH TimeBoundaries AS (
  SELECT
    -- Fixed 'Today's' start (used as the exclusive end boundary for 'yesterday' and 'last 7 days')
    TIMESTAMP(DATE('2024-12-03')) AS start_of_today,

    -- Yesterday boundaries (relative to fixed date)
    TIMESTAMP(DATE_SUB(DATE('2024-12-03'), INTERVAL 1 DAY)) AS start_of_yesterday, -- 2024-12-02 00:00:00

    -- Last 7 days boundaries (relative to fixed date, ending yesterday)
    TIMESTAMP(DATE_SUB(DATE('2024-12-03'), INTERVAL 7 DAY)) AS start_of_last_7_days, -- 2024-11-26 00:00:00

    -- Last Calendar Month boundaries (relative to fixed date: Nov 2024)
    TIMESTAMP(DATE_TRUNC(DATE_SUB(DATE('2024-12-03'), INTERVAL 1 MONTH), MONTH)) AS start_of_last_calendar_month, -- 2024-11-01 00:00:00
    TIMESTAMP(DATE_TRUNC(DATE('2024-12-03'), MONTH)) AS start_of_this_calendar_month, -- 2024-12-01 00:00:00 (Exclusive end boundary)

    -- Last Calendar Year boundaries (relative to fixed date: 2023)
    TIMESTAMP(DATE_TRUNC(DATE_SUB(DATE('2024-12-03'), INTERVAL 1 YEAR), YEAR)) AS start_of_last_calendar_year, -- 2023-01-01 00:00:00
    TIMESTAMP(DATE_TRUNC(DATE('2024-12-03'), YEAR)) AS start_of_this_calendar_year -- 2024-01-01 00:00:00 (Exclusive end boundary)
)

SELECT
  -- Use SUM(IF(...)) for conditional aggregation
  -- COALESCE ensures we get 0 instead of NULL if no sales occurred in a period

  -- Sales for Yesterday (relative to fixed date: 2024-12-02)
  COALESCE(SUM(IF(oi.created_at >= tb.start_of_yesterday AND oi.created_at < tb.start_of_today, oi.sale_price, 0)), 0) AS sales_last_day,

  -- Sales for Last 7 Days (relative to fixed date: 2024-11-26 to 2024-12-02)
  COALESCE(SUM(IF(oi.created_at >= tb.start_of_last_7_days AND oi.created_at < tb.start_of_today, oi.sale_price, 0)), 0) AS sales_last_7_days,

  -- Sales for Last Calendar Month (relative to fixed date: Nov 2024)
  COALESCE(SUM(IF(oi.created_at >= tb.start_of_last_calendar_month AND oi.created_at < tb.start_of_this_calendar_month, oi.sale_price, 0)), 0) AS sales_last_calendar_month,

  -- Sales for Last Calendar Year (relative to fixed date: 2023)
  COALESCE(SUM(IF(oi.created_at >= tb.start_of_last_calendar_year AND oi.created_at < tb.start_of_this_calendar_year, oi.sale_price, 0)), 0) AS sales_last_calendar_year

FROM
  order_items AS oi
CROSS JOIN -- There's only one row in TimeBoundaries, so CROSS JOIN is efficient
  TimeBoundaries AS tb
WHERE
  -- *** IMPORTANT FILTERING ***
  -- 1. Filter statuses that don't count as sales (adjust as needed)
  oi.status NOT IN ('Cancelled', 'Returned')

  -- 2. Pre-filter rows based on the widest time range needed for efficiency.
  --    Only consider items created from the start of the earliest period (last calendar year)
  --    up to the end of the latest period (end of yesterday = start of today).
  AND oi.created_at >= tb.start_of_last_calendar_year -- Start of 2023 based on fixed date
  AND oi.created_at < tb.start_of_today; -- Start of 2024-12-03 based on fixed date;
        """
    },
    "get_out_of_stock_items": {
      "kind": "spanner-sql",
      "source": "spanner-ecom-source",
      "description": "Use this tool to look up products that are out of stock.",
      "statement": """SELECT p.id, p.name 
FROM products p
LEFT JOIN inventory_items i ON p.id = i.product_id AND i.sold_at IS NULL
WHERE i.id IS NULL;
        """
    },
    "get_products_by_natural_language": {
      "kind": "spanner-sql",
      "source": "spanner-ecom-source",
      "description": "Use this tool to look up product details from the product catalog. Returns only in-stock items. Availble data includes cost, category, brand, name, retail_price, department, sku, and on_hand_count.",
      "statement": """WITH e AS (
    SELECT embeddings.values
    FROM ML.PREDICT(
    MODEL EmbeddingsModel, (
            SELECT @user_query as content
        )
    )
), on_hand AS (
  SELECT product_id, product_name, COUNT(*) AS on_hand_count FROM inventory_items i
  WHERE i.sold_at IS NULL 
  GROUP BY i.product_id, i.product_name
  HAVING COUNT(id) > 0
)
SELECT
  products.id,
  products.name,
  products.brand,
  products.department,
  products.cost,
  products.retail_price,
  products.sku,
  on_hand.on_hand_count
FROM products, e
JOIN on_hand ON products.id = on_hand.product_id
ORDER BY COSINE_DISTANCE(
    products.embedding,
    e.values
  )
LIMIT 10;
""",
      "parameters": [
        {
          "name": "user_query",
          "type": "string",
          "description": "user_query is a short description of the product, including brand. Example: 'Seven7 Women's Long Sleeve Stripe Belted Top'"
        }
      ]
    }
  },
  "toolsets": {
    "default-toolset": [
      "get_products_by_natural_language",
      "get_out_of_stock_items"
    ]
  }
}

with open("tools.yaml", "w") as file:
    file.write(json.dumps(tools_config))


#### Create Secret Version from tools.yaml

In [ ]:
# Create the secret
! gcloud secrets create tools --data-file=tools.yaml

# Or update if it already exists
! gcloud secrets versions add tools --data-file=tools.yaml

#### Delete Local Copy of tools.yaml

In [ ]:
os.remove('tools.yaml')

#### Deploy Toolbox to Cloud Run

In [ ]:
# Define Toolbox Container Image
image = 'us-central1-docker.pkg.dev/database-toolbox/toolbox/toolbox:latest'

# Deploy to Cloud Run
! gcloud run deploy toolbox \
    --image {image} \
    --service-account toolbox-identity \
    --region {region} \
    --set-secrets "/app/tools.yaml=tools:latest" \
    --args="--tools_file=/app/tools.yaml","--address=0.0.0.0","--port=8080" \
    --allow-unauthenticated # https://cloud.google.com/run/docs/authenticating/public#gcloud


#### Test the Tool

##### Manually Invoke the Tools

In [ ]:
from toolbox_langchain import ToolboxClient

# Set the toolbox_url
toolbox_url = f"https://toolbox-{project_number}.{region}.run.app/"

# Replace with your Toolbox service's URL
toolbox = ToolboxClient("https://toolbox-797738608815.us-central1.run.app")

# Load all tools
tools = toolbox.load_toolset()

# Test the tools
for t in tools:
  if t.name == 'get_products_by_natural_language':
    result = await t.arun({ "user_query": "Luxury items for men" })
    print(result)
  else:
    result = await t.arun({})
    print(result)

##### Invoke the Tools via LangGraph

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_google_vertexai import ChatVertexAI
from langgraph.checkpoint.memory import MemorySaver
from toolbox_langchain import ToolboxClient

prompt = ''' You're a helpful AI assistant. Select the best available Tool to get relevant data, then return the raw data and provide a summary answer the following question:\n '''

queries = [ "Which products are out of stock?",
           "What products are made by Seven7?",
            "What were total sales yesterday?"]

# Load the tools from the Toolbox server
client = ToolboxClient(toolbox_url)
tools = client.load_toolset()

model = ChatVertexAI(model="gemini-2.0-flash")

agent = create_react_agent(model, tools, checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "thread-1"}}
for query in queries:
    inputs = {"messages": [("user", prompt + query)]}
    response = agent.invoke(inputs, stream_mode="values", config=config)
    print(response["messages"][-1].content) # AI summary
    print(response["messages"][-2].content) # Tool data

## Setup Agentspace Cloud Run Proxy

### Create a Service Account and Add Necessary Permissions for Agentspace LangGraph Proxy

In [ ]:
# Reference: https://cloud.google.com/build/docs/deploying-builds/deploy-cloud-run#required_permissions
#            https://cloud.google.com/run/docs/deploying-source-code#required_roles

! gcloud iam service-accounts create agentspace-proxy

roles_array = [
    "roles/run.developer",
    "roles/logging.logWriter",
    "roles/artifactregistry.writer",
    "roles/iam.serviceAccountUser",
    "roles/storage.admin",
    "roles/aiplatform.user"
]

for r in roles_array:
  ! gcloud projects add-iam-policy-binding {project_id} \
      --member serviceAccount:agentspace-proxy@{project_id}.iam.gserviceaccount.com \
      --role="{r}"

### Add Additional Required Permissions for Cloud Build Service Account

In [ ]:
! gcloud projects add-iam-policy-binding {project_id} \
      --member=serviceAccount:{project_number}-compute@developer.gserviceaccount.com \
      --role=roles/run.builder

### Define the Cloud Run Function

#### requirements.txt

In [ ]:
! mkdir -p cloud-run-source

requirements = """functions-framework==3.*
langgraph==0.3.21
langchain-google-vertexai==2.0.18
toolbox-langchain==0.1.0
"""

with open("cloud-run-source/requirements.txt", "w") as file:
    file.write(requirements)

#### main.py

In [ ]:
function_definition = """import functions_framework
from langgraph.prebuilt import create_react_agent
from langchain_google_vertexai import ChatVertexAI
from langgraph.checkpoint.memory import MemorySaver
from toolbox_langchain import ToolboxClient
import json

@functions_framework.http
def invoke_toolbox(request):

    request_json = request.get_json(silent=True)
    request_args = request.args
    print("JSON:" + str(request_json))
    print("args:" + str(request_args))


    prompt = ''' You're a helpful AI assistant. You select the best Tool to get relevant data to answer the User's questions. '''

    user_query = request_json["user_query"]

    # Load the tools from the Toolbox server
    client = ToolboxClient('TOOLBOX_URL')
    tools = client.load_toolset()

    model = ChatVertexAI(model="gemini-2.0-flash")

    agent = create_react_agent(model, tools, checkpointer=MemorySaver())

    config = {"configurable": {"thread_id": "thread-1"}}

    inputs = {"messages": [("user", prompt + user_query)]}
    response = agent.invoke(inputs, stream_mode="values", config=config)

    response_message = {}
    response_message['ai_summary'] = response["messages"][-1].content
    response_message['tool_data'] = json.loads(response["messages"][-2].content)

    return {"message": response_message}
"""

function_definition = function_definition.replace("TOOLBOX_URL", toolbox_url)

with open("cloud-run-source/main.py", "w") as file:
    file.write(function_definition)

### Deploy the Agentspace LangGraph Proxy Cloud Run Function

In [ ]:
! gcloud functions deploy agentspace-toolbox-proxy \
  --gen2 \
  --region={region} \
  --runtime=python312 \
  --source="./cloud-run-source" \
  --entry-point="invoke_toolbox" \
  --run-service-account="agentspace-proxy@{project_id}.iam.gserviceaccount.com" \
  --service-account="agentspace-proxy@{project_id}.iam.gserviceaccount.com" \
  --trigger-http \
  --allow-unauthenticated \
  --memory=2gi

### Test the Agentspace LangGraph Proxy

In [ ]:
# Set the proxy url
agentspace_proxy_url = f"https://agentspace-toolbox-proxy-{project_number}.{region}.run.app"

# Test the natural language question tool
request_body = {"user_query": "What handbag styles are in stock?"}
response = rest_api_helper(authed_session, agentspace_proxy_url, 'POST', request_body, {})
print(response)


# Test the sales total tool
request_body = {"user_query": "What were total sales yesterday?"}
response = rest_api_helper(authed_session, agentspace_proxy_url, 'POST', request_body, {})
print(response)


# Test the out of stock tool
request_body = {"user_query": "Which items are out of stock?"}
response = rest_api_helper(authed_session, agentspace_proxy_url, 'POST', request_body, {})
print(response)

## Setup Agentspace

### Accept Terms

In [ ]:
# https://cloud.google.com/generative-ai-app-builder/docs/reference/rest/v1/projects/provision

url = f"https://discoveryengine.googleapis.com/v1/projects/{project_id}:provision"
request_body = {
  "acceptDataUseTerms": "true",
  "dataUseTermsVersion": "2022-11-23"
}
parameters = {}
result = rest_api_helper(authed_session, url, 'POST', request_body, parameters)
result

### Create Data Store

In [ ]:
# Reference: https://cloud.google.com/generative-ai-app-builder/docs/samples/genappbuilder-create-data-store

from google.api_core.client_options import ClientOptions
from google.cloud import discoveryengine

location = "global"
data_store_id = "cymbal_shops_datastore"


def create_data_store(
    project_id: str,
    location: str,
    data_store_id: str,
) -> str:
    #  For more information, refer to:
    # https://cloud.google.com/generative-ai-app-builder/docs/locations#specify_a_multi-region_for_your_data_store
    client_options = (
        ClientOptions(api_endpoint=f"{location}-discoveryengine.googleapis.com")
        if location != "global"
        else None
    )

    # Create a client
    client = discoveryengine.DataStoreServiceClient(client_options=client_options)

    # The full resource name of the collection
    # e.g. projects/{project}/locations/{location}/collections/default_collection
    parent = client.collection_path(
        project=project_id,
        location=location,
        collection="default_collection",
    )

    data_store = discoveryengine.DataStore(
        display_name="Cymbal Shops Data Store",
        # Options: GENERIC, MEDIA, HEALTHCARE_FHIR
        industry_vertical=discoveryengine.IndustryVertical.GENERIC,
        # Options: SOLUTION_TYPE_RECOMMENDATION, SOLUTION_TYPE_SEARCH, SOLUTION_TYPE_CHAT, SOLUTION_TYPE_GENERATIVE_CHAT
        solution_types=[discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH],
        # TODO(developer): Update content_config based on data store type.
        # Options: NO_CONTENT, CONTENT_REQUIRED, PUBLIC_WEBSITE
        content_config=discoveryengine.DataStore.ContentConfig.CONTENT_REQUIRED,
    )

    request = discoveryengine.CreateDataStoreRequest(
        parent=parent,
        data_store_id=data_store_id,
        data_store=data_store,
        # Optional: For Advanced Site Search Only
        # create_advanced_site_search=True,
    )

    # Make the request
    operation = client.create_data_store(request=request)

    print(f"Waiting for operation to complete: {operation.operation.name}")
    response = operation.result()

    # After the operation is complete,
    # get information from operation metadata
    metadata = discoveryengine.CreateDataStoreMetadata(operation.metadata)

    # Handle the response
    print(response)
    print(metadata)

    return operation.operation.name

result = create_data_store(project_id, location, data_store_id)
result


### Import Data Store Documents from GCS

In [ ]:
# Reference: https://cloud.google.com/generative-ai-app-builder/docs/samples/genappbuilder-import-documents-spanner

from google.api_core.client_options import ClientOptions
from google.cloud import discoveryengine

location = "global"

#  For more information, refer to:
# https://cloud.google.com/generative-ai-app-builder/docs/locations#specify_a_multi-region_for_your_data_store
client_options = (
    ClientOptions(api_endpoint=f"{location}-discoveryengine.googleapis.com")
    if location != "global"
    else None
)

# Create a client
client = discoveryengine.DocumentServiceClient(client_options=client_options)

# The full resource name of the search engine branch.
# e.g. projects/{project}/locations/{location}/dataStores/{data_store_id}/branches/{branch}
parent = client.branch_path(
    project=project_id,
    location=location,
    data_store=data_store_id,
    branch="default_branch",
)

gcs_uris = [
    'gs://pr-public-demo-data/spanner-retail-demo/cymbal-shops-data-store-docs/Cymbal Shops - Company Overview.pdf',
    'gs://pr-public-demo-data/spanner-retail-demo/cymbal-shops-data-store-docs/NRF _ 25 predictions for the retail industry in 2025.pdf'
]

request = discoveryengine.ImportDocumentsRequest(
    parent=parent,
    gcs_source=discoveryengine.GcsSource(
      input_uris=gcs_uris,
      # Options:
        # - `content` - Unstructured documents (PDF, HTML, DOC, TXT, PPTX)
        # - `custom` - Unstructured documents with custom JSONL metadata
        # - `document` - Structured documents in the discoveryengine.Document format.
        # - `csv` - Unstructured documents with CSV metadata
        data_schema="content",
    ),
    # Options: `FULL`, `INCREMENTAL`
    reconciliation_mode=discoveryengine.ImportDocumentsRequest.ReconciliationMode.FULL,
)

# Make the request
operation = client.import_documents(request=request)

# Handle the response
print("Kicked off import of Cymbal Shops Data Store data.")


### Create an Agentspace Conversational app

In [ ]:
# Reference: https://cloud.google.com/dialogflow/cx/docs/reference/rest/v3beta1/projects.locations.agents/create?apix_params=%7B%22parent%22%3A%22projects%2F%7Bproject_id%7D%2Flocations%2Fglobal%22%2C%22resource%22%3A%7B%7D%7D
#            https://cloud.google.com/dialogflow/cx/docs/reference/rest/v3beta1/projects.locations.agents#Agent

url = f"https://dialogflow.googleapis.com/v3beta1/projects/{project_id}/locations/global/agents"
request_body = {
    "display_name": "Toolbox Agent",
    "default_language_code": "en",
    "time_zone": "America/Chicago",
    "start_playbook": f"projects/{project_id}/locations/glocal/agents/*/playbooks/00000000-0000-0000-0000-000000000000"
}
parameters = {}

result = rest_api_helper(authed_session, url, 'POST', request_body, parameters)
agent_id = result['name']
result

### Create an Agentspace Tool

In [ ]:
# Reference: https://cloud.google.com/dialogflow/cx/docs/reference/rest/v3beta1/projects.locations.agents.tools/create
#            https://cloud.google.com/dialogflow/cx/docs/reference/rest/v3beta1/projects.locations.agents.tools#Tool

url = f"https://dialogflow.googleapis.com/v3beta1/{agent_id}/tools"
request_body = {
    "display_name": "Database Toolbox",
    "description": "A tool to lookup product data, inventory levels, and sales figures",
    "tool_type": "CUSTOMIZED_TOOL",
    "open_api_spec": {
        "text_schema": f"""openapi: 3.0.0
info:
  title: GenAI Toolbox API
  version: 1.0.0
servers:
  - url: '{agentspace_proxy_url}'
paths:
  /:
    post:
      summary: Lookup product data, inventory levels, or sales figures
      operationId: searchDatabase
      requestBody:
        description: Database search
        required: true
        content:
          application/json:
            schema:
              $ref: '#/components/schemas/DatabaseSearch'
      responses:
        '200':
          description: Success
components:
  schemas:
    DatabaseSearch:
      type: object
      required:
        - user_query
      properties:
        user_query:
          type: string
"""
    }
}
parameters = {}

response = rest_api_helper(authed_session, url, 'POST', request_body, parameters)
agentspace_tool_id = response['name']
response

### Add permissions for Agentspace Tool to invoke Cloud Run

In [ ]:
roles_array = [
    "roles/run.invoker",
]

for r in roles_array:
  ! gcloud projects add-iam-policy-binding {project_id} \
      --member="serviceAccount:service-{project_number}@gcp-sa-dialogflow.iam.gserviceaccount.com" \
      --role="{r}"

### Create a Conversational Agent Playbook

In [ ]:
# Update the default playbook
url = f"https://dialogflow.googleapis.com/v3beta1/{agent_id}/playbooks/00000000-0000-0000-0000-000000000000"
request_body = {
    "displayName": "Use a Tool",
    "goal": "Use a tool to get more context to answer the user's question.",
    "referenced_tools": f"{agentspace_tool_id}",
    "instruction": {
        "guidelines": "Don't make anything up. Only provide factual information grounded in data provided by Tools and Data Sources. Always be polite and professional.",
        "steps": [
            {"text": "Use ${TOOL:Database Toolbox}. Don't ask clarifying questions - just immediately send the user's query to the tool."},
            {"text": "Respond to the user with a detailed response based on the output of the Tool."},
        ]
    }

}
parameters = {}

response = rest_api_helper(authed_session, url, 'PATCH', request_body, parameters)
response

## Manual Steps

The remaining steps require manual configuration in the console due to lack of API coverage this new Agentspace functionality. Instructions are taken from [this Qwiklab](https://partner.cloudskillsboost.google/course_templates/1191/labs/525477).

### Create an Agentspace app

In this task, you'll create a new Agentspace app using the "Agentspace" template and integrating Google as the Identity Provider, while linking to a data store.

1. Navigate to **Agentspace**. On the Agentspace landing page, select **Manage** in the **Agentspace** tile (not NotebookLM for Enterprise).

2. Select **Apps** > **+ Create App**.

3. Find the **Agentspace** card and click **CREATE** to create an Agentspace app.

4. For **app name**, enter `Agentspace`

5. For **company name**, enter `Cymbal Shops`

6. Keep the location set to **global**.

7. Under **select tier** choose **Search + Assistant**.

8. Click **Continue**.

9. For an **Identity provider**, click **SELECT** on the **Google Identity Provider** card.

10. On the **Data** pane, select **Cymbal Shops Data Store**.

11. Click **Create**.

### Integrate your conversational agent with your Agentspace app

In this task, you'll grant your Agentspace assistant the ability to send messages to your conversational agent and receive its responses.

1. Navigate to **Agentspace**. On the Agentspace landing page, select **Manage** in the **Agentspace** tile (not NotebookLM for Enterprise). 

2. Select **Apps** > `Agentspace` App.

3. From the left-hand navigation, select **Configurations**.

4. Select the **Assistant** tab.

5. Under the **Agents** header, select **Add an Item**. A card will be displayed to connect a **New Agent**:

6. Select your browser tab displaying your **Conversational Agents** console.

7. From the **Agent** dropdown at the top of the console, select **View all agents**.

8. At the end of your `Toolbox Agent` agent's row, select the **Options icon** (three vertical dots) and select **Copy name**.

9. Navigate back to your **Agentspace** tab, and in the **New Agent** card, paste the copied value in the **Agent** field.

10. For an **Agent display** name, use `Toolbox Agent`.

11. For Instructions enter:

  ```
    Use this tool to lookup sales, product, and inventory information. 
  ```

12. Notice that these instructions instruct the Agentspace assistant to do the work of gathering the required information before passing the details to the conversational agent for a single turn of conversation.

13. Click **Done**.

14. Click **Save and Publish** at the bottom of the pane.

### Communicate with your conversational agent through the Agentspace assistant

In this task, your Agentspace assistant will be able to communicate with the conversational agent, which will then utilize its tool to lookup product data.

> NOTE: It can take up to 10 minutes for your Agentspace app to be created. You can try the steps below, but if they don't proceed as expected, try to give your app more time to be created.

> NOTE: The Cloud Run instance deployed in this notebook are configured to scale down to zero when unused to save on cost. This can cause the first query to take a while to run. Subsequent queries should run faster.

1. Select your **Agentspace**. Navigate to the **Integration** tab from the left-hand navigation menu.

2. Under **The link to your web app** header, click **Open**. As stated at the start of this task, if you see a 404 error, you may need to give your app more time to be created. You can reload the page every few minutes until the Agentspace web app appears.

3. In the primary search bar, enter: `What were our total sales last year?`

4. Your chat will be saved as a **Conversation** under the **Recents** header on the left-hand menu of the Agentspace web app.

5. Your assistant should have responded to your request. If you are asked any additional questions, use the Ask a follow-up field to reply to the assistant.

6. Under the assistant's responses, there is an **Options menu** (three vertical dots). Expand it and select **Show diagnostic info**.

7. In the diagnostic info displayed, you can view the metadata of the response, which includes `"functionName": "Toolbox_Agent"`. This confirms that the Agentspace assistant invoked your conversational agent as a function call and received a response from it, which it has passed back to you.

8. Try asking other questions about products and inventory that will use our GenAI Toolbox Integration. For example, you could ask:

    - `Which CK items do we have in stock?`
    - `Which items are currently out of stock?` 
    - `Which Fossil watches do we carry?`

9. Please note, in this activity you used very minimal Playbook instructions and no conversation examples, which means that this agent will not be very robust. If you need to restart the conversation to try it again, click the **New Conversation** button in the upper left.

## References

* [Google Agentspace](https://cloud.google.com/products/agentspace?e=48754805&hl=en)
* [GenAI Toolbox](https://github.com/googleapis/genai-toolbox)